In [2]:
import numpy as np                                  # 导入数值计算库numpy

# 计算三维杆单元刚度矩阵
def truss3d_element_stiffness(x1, x2, E, A):        # 定义刚度矩阵函数
    x1 = np.array(x1, dtype=float)                 # 节点1坐标转为浮点数组
    x2 = np.array(x2, dtype=float)                 # 节点2坐标转为浮点数组
    dx = x2[0] - x1[0]                             # x方向坐标差值
    dy = x2[1] - x1[1]                             # y方向坐标差值
    dz = x2[2] - x1[2]                             # z方向坐标差值
    L = np.sqrt(dx**2 + dy**2 + dz**2)             # 计算单元长度L

    if L < 1e-12:                                  # 判断单元是否退化
        raise ValueError("错误：节点重合，单元退化！")  # 节点重合报错

    cx = dx / L                                    # x方向方向余弦
    cy = dy / L                                    # y方向方向余弦
    cz = dz / L                                    # z方向方向余弦
    G = np.array([-cx, -cy, -cz, cx, cy, cz])      # 构造位移转换向量G
    Ke = (E * A / L) * np.outer(G, G)              # 计算6×6刚度矩阵Ke
    return L, np.array([cx, cy, cz]), Ke           # 返回长度、方向余弦、Ke

# 由位移计算应变、应力、轴力
def truss3d_element_stress(x1, x2, E, A, de):      # 定义应力计算函数
    L, c, _ = truss3d_element_stiffness(x1, x2, E, A)  # 获取L与方向余弦
    cx, cy, cz = c                                 # 解包方向余弦
    de = np.array(de, dtype=float)                 # 位移转为浮点数组
    epsilon = (-cx*de[0]-cy*de[1]-cz*de[2]+cx*de[3]+cy*de[4]+cz*de[5])/L  # 计算应变
    sigma = E * epsilon                            # 胡克定律计算应力
    N = sigma * A                                  # 由应力计算轴力
    return epsilon, sigma, N                       # 返回应变、应力、轴力

# 检验刚度矩阵性质
def check_matrix_properties(Ke):                   # 定义矩阵性质检验函数
    is_symmetric = np.allclose(Ke, Ke.T)          # 判断是否对称
    det = np.linalg.det(Ke)                       # 计算行列式
    is_singular = abs(det) < 1e-6                 # 判断是否奇异
    eig_vals = np.linalg.eigvals(Ke)              # 计算特征值
    is_positive_semi = np.all(eig_vals >= -1e-6)  # 判断是否半正定

    print("单元刚度矩阵性质检验")        # 打印标题
    print(f"对称：{is_symmetric}")                # 输出对称性
    print(f"奇异：{is_singular}")                # 输出奇异性
    print(f"半正定：{is_positive_semi}")         # 输出半正定性
    print(f"行列式：{det:.2e}")                  # 输出行列式
    print(f"最小特征值：{np.min(eig_vals):.2e}") # 输出最小特征值
    print("============================\n")       # 打印结束行

# ===================== 算例1：沿x轴杆单元 =====================
print("="*50)                                      # 打印分隔线
print("                算例 1：沿x轴杆单元")      # 打印算例1标题
print("="*50)                                      # 打印分隔线
x1 = [0,0,0]                                       # 节点1坐标
x2 = [2,0,0]                                       # 节点2坐标
E = 200e9                                          # 弹性模量200GPa
A = 1.0e-4                                         # 横截面积
de = [0,0,0, 1e-3, 0, 0]                           # 节点位移向量

L1, c1, Ke1 = truss3d_element_stiffness(x1,x2,E,A) # 计算算例1刚度相关
eps1, sig1, N1 = truss3d_element_stress(x1,x2,E,A,de) # 计算算例1应力相关

print(f"单元长度 L = {L1:.2f} m")                 # 输出单元长度
# 一维杆只输出x方向方向余弦
print(f"方向余弦 c = [{c1[0]:.0f}]")
print(f"应变 ε = {eps1:.2e}")                     # 输出应变
print(f"应力 σ = {sig1/1e6:.2f} MPa")             # 输出应力(MPa)
print(f"轴力 N = {N1:.2e} N")                     # 输出轴力

# 一维杆输出2×2刚度矩阵，不再输出6×6
print("\n===== 算例1 全局刚度矩阵 Ke1 (2×2) =====")
k_1d = E * A / L1
Ke1_22 = np.array([[k_1d, -k_1d],[-k_1d, k_1d]])
np.set_printoptions(precision=4, suppress=True)
print(Ke1_22)
check_matrix_properties(Ke1)                       # 检验矩阵性质

print("=== 算例1：刚度矩阵退化为一维杆验证 ===") # 打印验证标题
print("一维杆理论刚度 EA/L =", k_1d)              # 输出理论值
print("退化验证通过：", np.allclose(Ke1_22, [[k_1d,-k_1d],[-k_1d,k_1d]]),"\n") # 验证结果

# ===================== 算例2：空间任意方向杆单元 =====================
print("="*50)                                      # 打印分隔线
print("              算例 2：空间任意方向杆单元") # 打印算例2标题
print("="*50)                                      # 打印分隔线
x1 = [0,0,0]                                       # 节点1坐标
x2 = [1,2,2]                                       # 节点2坐标
E = 210e9                                          # 弹性模量210GPa
A = 2.0e-4                                         # 横截面积
de = [0,0,0, 1e-3, 2e-3, 2e-3]                     # 节点位移向量

L2, c2, Ke2 = truss3d_element_stiffness(x1,x2,E,A) # 计算算例2刚度相关
eps2, sig2, N2 = truss3d_element_stress(x1,x2,E,A,de) # 计算算例2应力相关

print(f"单元长度 L = {L2:.2f} m")                 # 输出单元长度
print(f"方向余弦 c = {c2}")                       # 输出方向余弦
print(f"应变 ε = {eps2:.2e}")                     # 输出应变
print(f"应力 σ = {sig2/1e6:.2f} MPa")             # 输出应力(MPa)
print(f"轴力 N = {N2:.2e} N")                     # 输出轴力

print("\n===== 算例2 全局刚度矩阵 Ke2 (6×6) =====") # 打印刚度矩阵标题
print(Ke2)                                         # 输出刚度矩阵
check_matrix_properties(Ke2)                       # 检验矩阵性质


# ===================== 刚体位移检验 =====================
print("="*50)                                      # 打印分隔线
print("             刚体位移检验（无应力/轴力）")  # 打印刚体位移标题
print("="*50)                                      # 打印分隔线
de_rigid = [0.1, 0.2, 0.3, 0.1, 0.2, 0.3]          # 刚体位移（两节点相同）
eps_r, sig_r, N_r = truss3d_element_stress(x1,x2,E,A,de_rigid) # 计算刚体位移下内力
print(f"刚体位移应变：{eps_r:.2e}")               # 输出应变
print(f"刚体位移应力：{sig_r:.2e} Pa")             # 输出应力
print(f"刚体位移轴力：{N_r:.2e} N")               # 输出轴力
print("刚体位移不产生内力 \n")                   # 输出结论

# ===================== 刚度矩阵物理意义验证 =====================
print("="*50)                                      # 打印分隔线
print("          物理意义：j自由度=1，其余=0，Fe=Ke·de") # 打印物理意义标题
print("="*50)                                      # 打印分隔线
j = 2                                              # 选定第3个自由度(u2)
de_unit = np.zeros(6)                              # 构造6维0位移向量
de_unit[j] = 1.0                                  # 第j自由度设为单位位移1
Fe = Ke2 @ de_unit                                # 计算节点力Fe=Ke*de
print(f"第 {j} 自由度单位位移，力向量 Fe：")       # 打印Fe标题
print(np.round(Fe, 2))                             # 输出Fe向量
print(f"\n刚度矩阵第 {j} 列：")                    # 打印Ke列标题
print(np.round(Ke2[:, j], 2))                      # 输出Ke第j列
print("\n结论：Fe 等于刚度矩阵第 j 列 ")         # 输出结论


                算例 1：沿x轴杆单元
单元长度 L = 2.00 m
方向余弦 c = [1]
应变 ε = 5.00e-04
应力 σ = 100.00 MPa
轴力 N = 1.00e+04 N

===== 算例1 全局刚度矩阵 Ke1 (2×2) =====
[[ 10000000. -10000000.]
 [-10000000.  10000000.]]
单元刚度矩阵性质检验
对称：True
奇异：True
半正定：True
行列式：0.00e+00
最小特征值：0.00e+00

=== 算例1：刚度矩阵退化为一维杆验证 ===
一维杆理论刚度 EA/L = 10000000.0
退化验证通过： True 

              算例 2：空间任意方向杆单元
单元长度 L = 3.00 m
方向余弦 c = [0.3333 0.6667 0.6667]
应变 ε = 1.00e-03
应力 σ = 210.00 MPa
轴力 N = 4.20e+04 N

===== 算例2 全局刚度矩阵 Ke2 (6×6) =====
[[ 1555555.5556  3111111.1111  3111111.1111 -1555555.5556 -3111111.1111
  -3111111.1111]
 [ 3111111.1111  6222222.2222  6222222.2222 -3111111.1111 -6222222.2222
  -6222222.2222]
 [ 3111111.1111  6222222.2222  6222222.2222 -3111111.1111 -6222222.2222
  -6222222.2222]
 [-1555555.5556 -3111111.1111 -3111111.1111  1555555.5556  3111111.1111
   3111111.1111]
 [-3111111.1111 -6222222.2222 -6222222.2222  3111111.1111  6222222.2222
   6222222.2222]
 [-3111111.1111 -6222222.2222 -6222222.2222  3111111.1111  6222222.